# Portkey Agent Gateway — Live Demo

Portkey makes **any AI agent framework production-ready** by routing all LLM calls through its AI Gateway. You get observability, reliability, and cost tracking for every agent step — without changing your agent code.

```
┌──────────────────────────────────────────────────┐
│              Your Agent Framework                │
│  (OpenAI Agents / CrewAI / LangGraph / Strands)  │
└───────────────────┬──────────────────────────────┘
                    │  LLM calls
                    ▼
┌──────────────────────────────────────────────────┐
│           Portkey AI Gateway                     │
│  ┌──────────────────────────────────────────┐    │
│  │ Observability  │ Traces, costs, latency  │    │
│  │ Reliability    │ Fallbacks, retries, LB   │    │
│  │ Guardrails     │ Input/output validation  │    │
│  │ Interop        │ Swap LLMs via config     │    │
│  └──────────────────────────────────────────┘    │
└───────────────────┬──────────────────────────────┘
                    │
        ┌───────────┼───────────┐
        ▼           ▼           ▼
     OpenAI    Anthropic    Gemini   ... (250+ providers)
```

### What we'll cover

| # | Section | Framework | What it shows |
|---|---------|-----------|---------------|
| 1 | **Setup** | — | Install deps, configure API keys |
| 2 | **OpenAI Agents SDK** | `openai-agents` | Global client, agent with tools, handoffs |
| 3 | **LangChain / LangGraph** | `langchain` | ChatOpenAI through Portkey, graph agent |
| 4 | **CrewAI** | `crewai` | Multi-agent crew through Portkey |
| 5 | **Strands Agents** | `strands-agents` | Native PortkeyStrands adapter |
| 6 | **Google ADK** | `google-adk` | Native PortkeyAdk adapter |
| 7 | **Auto-Instrumentation** | Any framework | One-line tracing for CrewAI / LangGraph |
| 8 | **Custom ReAct Agent** | Raw Portkey SDK | Build-your-own agent loop |
| 9 | **Reliability for Agents** | Any | Fallbacks + retries for long pipelines |
| 10 | **Observability** | Any | Traces, metadata, cost tracking |

> **Time:** ~30 minutes  
> **Prerequisites:** Python 3.9+, [Portkey account](https://app.portkey.ai), virtual key(s) for your LLM provider(s)

---
## 1. Setup

Install the Portkey SDK and your agent framework(s) of choice. You don't need all of them — install the ones you want to demo.

In [ ]:
# Core (always needed)
%pip install portkey-ai python-dotenv -q

# Agent frameworks (install the ones you need)
%pip install openai-agents -q               # Section 2: OpenAI Agents SDK
%pip install langchain langchain-openai -q   # Section 3: LangChain / LangGraph
# %pip install langgraph -q                  # Section 3: LangGraph (optional)
# %pip install crewai -q                     # Section 4: CrewAI
# %pip install 'portkey-ai[strands]' -q      # Section 5: Strands Agents
# %pip install 'portkey-ai[adk]' -q          # Section 6: Google ADK

In [ ]:
import os
import json
import time
from dotenv import load_dotenv

load_dotenv()

PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")

# ----- Replace with your virtual key slug(s) -----
# Virtual keys wrap provider credentials (create at app.portkey.ai/virtual-keys)
VIRTUAL_KEY = "@your-virtual-key"           # e.g. "@my-openai"
MODEL        = f"{VIRTUAL_KEY}/gpt-4o-mini"  # primary model
BACKUP_MODEL = f"{VIRTUAL_KEY}/gpt-3.5-turbo"  # fallback model
# --------------------------------------------------

if not PORTKEY_API_KEY:
    raise EnvironmentError(
        "PORTKEY_API_KEY not found. "
        "Create a .env file with: PORTKEY_API_KEY=your_key_here"
    )

masked = PORTKEY_API_KEY[:8] + "*" * 20
print(f"PORTKEY_API_KEY: {masked}")
print(f"Primary model:  {MODEL}")
print(f"Backup model:   {BACKUP_MODEL}")
print("\nReady!")

---
## 2. OpenAI Agents SDK

The [OpenAI Agents SDK](https://github.com/openai/openai-agents-python) provides a lightweight framework for building agents. Portkey integrates by replacing the default OpenAI client with one that routes through the gateway.

### Three integration approaches

| Approach | Scope | Use case |
|----------|-------|----------|
| **Global client** | All agents | Simplest — one line of setup |
| **Custom provider** | Selective agents | Different configs per agent |
| **Per-agent client** | Individual agent | Fine-grained control |

We'll use the global client approach:

In [ ]:
from agents import Agent, Runner, set_default_openai_client, set_default_openai_api
from openai import AsyncOpenAI
from portkey_ai import PORTKEY_GATEWAY_URL, createHeaders

# Route all agent LLM calls through Portkey
portkey_client = AsyncOpenAI(
    base_url=PORTKEY_GATEWAY_URL,
    api_key=PORTKEY_API_KEY,
    default_headers=createHeaders(
        provider=VIRTUAL_KEY,
        trace_id="openai-agents-demo",
    ),
)

set_default_openai_client(portkey_client, use_for_tracing=False)
set_default_openai_api("chat_completions")

print("OpenAI Agents SDK configured to use Portkey gateway.")
print(f"  Gateway URL: {PORTKEY_GATEWAY_URL}")

In [ ]:
# 2a. Simple agent
agent = Agent(
    name="Helpful Assistant",
    instructions="You are a concise assistant. Answer in 1-2 sentences.",
    model="gpt-4o-mini",
)

result = Runner.run_sync(agent, "What is an AI Gateway?")
print(f"Agent: {result.final_output}")

In [ ]:
# 2b. Agent with a tool
from agents import function_tool
import random

@function_tool
def roll_dice(sides: int = 6) -> str:
    """Roll a die with the given number of sides."""
    result = random.randint(1, sides)
    return f"Rolled a {result} on a d{sides}"

dice_agent = Agent(
    name="Dice Roller",
    instructions="You help users roll dice. Use the roll_dice tool when asked.",
    model="gpt-4o-mini",
    tools=[roll_dice],
)

result = Runner.run_sync(dice_agent, "Roll a d20 for me!")
print(f"Agent: {result.final_output}")

In [ ]:
# 2c. Agent handoff (multi-agent)
spanish_agent = Agent(
    name="Spanish Translator",
    instructions="You translate everything to Spanish. Reply only in Spanish.",
    model="gpt-4o-mini",
)

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You are a triage agent. If the user wants a translation, "
        "hand off to the Spanish Translator. Otherwise, answer directly."
    ),
    model="gpt-4o-mini",
    handoffs=[spanish_agent],
)

result = Runner.run_sync(triage_agent, "Translate 'Hello world' to Spanish")
print(f"Final agent: {result.last_agent.name}")
print(f"Output: {result.final_output}")

---
## 3. LangChain / LangGraph

LangChain integrates with Portkey through `ChatOpenAI` — just point `base_url` to Portkey and pass your credentials in `default_headers`.

Every chain invocation, tool call, and retrieval step flows through the gateway.

In [ ]:
from langchain_openai import ChatOpenAI
from portkey_ai import createHeaders, PORTKEY_GATEWAY_URL

llm = ChatOpenAI(
    api_key="x",  # required by langchain but unused (Portkey handles auth)
    base_url=PORTKEY_GATEWAY_URL,
    default_headers=createHeaders(
        api_key=PORTKEY_API_KEY,
        provider=VIRTUAL_KEY,
        trace_id="langchain-demo",
        metadata={
            "framework": "langchain",
            "_user": "demo-user",
        },
    ),
    model="gpt-4o-mini",
)

print("LangChain ChatOpenAI configured with Portkey gateway.")

In [ ]:
# 3a. Simple chain invocation
response = llm.invoke("What is Portkey AI? Answer in one sentence.")
print(f"Response: {response.content}")

In [ ]:
# 3b. Chain with prompt template
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Be concise."),
    ("user", "{question}"),
])

chain = prompt | llm

result = chain.invoke({"role": "Python expert", "question": "What's the GIL?"})
print(f"Response: {result.content}")

In [ ]:
# 3c. LangChain with tool calling
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

llm_with_tools = llm.bind_tools([multiply, add])

result = llm_with_tools.invoke("What is 6 times 7?")
print(f"Content: {result.content}")
if result.tool_calls:
    for tc in result.tool_calls:
        print(f"Tool call: {tc['name']}({tc['args']})")

In [ ]:
# 3d. LangGraph agent (requires: pip install langgraph)
# Uncomment to run if langgraph is installed

# from typing import Annotated
# from typing_extensions import TypedDict
# from langgraph.graph import StateGraph
# from langgraph.graph.message import add_messages
#
# class State(TypedDict):
#     messages: Annotated[list, add_messages]
#
# graph_builder = StateGraph(State)
#
# def chatbot(state: State):
#     return {"messages": [llm.invoke(state["messages"])]}
#
# graph_builder.add_node("chatbot", chatbot)
# graph_builder.set_entry_point("chatbot")
# graph_builder.set_finish_point("chatbot")
# graph = graph_builder.compile()
#
# result = graph.invoke({"messages": [{"role": "user", "content": "Hello!"}]})
# print(f"Response: {result['messages'][-1].content}")

---
## 4. CrewAI

CrewAI lets you build multi-agent teams ("crews") that collaborate. Portkey adds observability and reliability to every LLM call in the crew.

Integration: pass `base_url` and `extra_headers` to CrewAI's `LLM` class.

In [ ]:
# Requires: pip install crewai
# Uncomment to run

# from crewai import Agent as CrewAgent, Task, Crew, LLM
# from portkey_ai import createHeaders, PORTKEY_GATEWAY_URL
#
# crew_llm = LLM(
#     model="gpt-4o-mini",
#     base_url=PORTKEY_GATEWAY_URL,
#     api_key="x",
#     extra_headers=createHeaders(
#         api_key=PORTKEY_API_KEY,
#         virtual_key=VIRTUAL_KEY,
#         trace_id="crewai-demo",
#     ),
# )
#
# researcher = CrewAgent(
#     role="Research Analyst",
#     goal="Find key facts about AI Gateways",
#     backstory="You are a senior technology analyst.",
#     llm=crew_llm,
#     verbose=True,
# )
#
# writer = CrewAgent(
#     role="Technical Writer",
#     goal="Write a concise summary from research findings",
#     backstory="You are a technical writer who makes complex topics simple.",
#     llm=crew_llm,
#     verbose=True,
# )
#
# research_task = Task(
#     description="Research what AI Gateways are and list 3 key benefits.",
#     expected_output="A list of 3 key benefits of AI Gateways.",
#     agent=researcher,
# )
#
# write_task = Task(
#     description="Write a 2-sentence summary based on the research.",
#     expected_output="A 2-sentence summary.",
#     agent=writer,
# )
#
# crew = Crew(
#     agents=[researcher, writer],
#     tasks=[research_task, write_task],
#     verbose=True,
# )
#
# result = crew.kickoff()
# print(f"\nCrew result:\n{result}")

print("CrewAI section — uncomment to run (requires: pip install crewai)")

---
## 5. Strands Agents

Portkey has a **native `PortkeyStrands` adapter** that implements the Strands `Model` interface. No need to configure OpenAI clients — it works directly.

In [ ]:
# Requires: pip install 'portkey-ai[strands]'
# Uncomment to run

# from portkey_ai.integrations.strands import PortkeyStrands
# from strands import Agent as StrandsAgent
#
# model = PortkeyStrands(
#     api_key=PORTKEY_API_KEY,
#     virtual_key=VIRTUAL_KEY,
#     model_id="gpt-4o-mini",
# )
#
# agent = StrandsAgent(model=model)
# response = agent("What is 2 + 2? Just the number.")
# print(f"Agent: {response}")

print("Strands section — uncomment to run (requires: pip install 'portkey-ai[strands]')")

---
## 6. Google ADK

Portkey also has a **native `PortkeyAdk` adapter** for Google's Agent Development Kit. It implements ADK's `BaseLlm` interface with full streaming and tool-calling support.

In [ ]:
# Requires: pip install 'portkey-ai[adk]'
# Uncomment to run

# from portkey_ai.integrations.adk import PortkeyAdk
# from google.adk import Agent as AdkAgent
# from google.adk.runners import Runner as AdkRunner
#
# model = PortkeyAdk(
#     model="gpt-4o-mini",
#     api_key=PORTKEY_API_KEY,
#     virtual_key=VIRTUAL_KEY,
# )
#
# agent = AdkAgent(
#     name="assistant",
#     model=model,
#     instruction="You are a helpful assistant. Be concise.",
# )
#
# # Run with ADK's runner
# runner = AdkRunner(agent=agent, app_name="portkey_demo")
# # ... see ADK docs for full usage

print("Google ADK section — uncomment to run (requires: pip install 'portkey-ai[adk]')")

---
## 7. Auto-Instrumentation (Beta)

For supported frameworks (CrewAI, LangGraph), Portkey can automatically capture traces with **one line of code** — no config changes to your agents.

Under the hood, it uses OpenTelemetry to patch the framework's methods and export spans to Portkey.

```python
from portkey_ai import Portkey
Portkey(api_key="...", instrumentation=True)
# That's it. All CrewAI/LangGraph calls are now traced.
```

### Supported frameworks
| Framework | What's instrumented |
|-----------|--------------------|
| **CrewAI** | `kickoff`, `execute_task`, tool calls, RAG operations |
| **LangGraph** | `StateGraph` construction, node execution, edges |

In [ ]:
from portkey_ai import Portkey

# This single line instruments all supported frameworks
# Requires: pip install 'portkey-ai[instrumentation]'
try:
    portkey = Portkey(api_key=PORTKEY_API_KEY, instrumentation=True)
    print("Auto-instrumentation enabled!")
    print("Any CrewAI or LangGraph code will now send traces to Portkey.")
    print("View them at: https://app.portkey.ai → Logs")
except ImportError as e:
    print(f"Instrumentation requires OpenTelemetry: {e}")
    print("Install with: pip install 'portkey-ai[instrumentation]'")

---
## 8. Custom ReAct Agent (Raw Portkey SDK)

Build your own agent loop with just the Portkey SDK. This shows the pattern every framework uses under the hood:

```
User message
    ↓
LLM (via Portkey) → decides: answer or call tool?
    ↓                           ↓
  Answer                    Execute tool
                                ↓
                    Tool result → back to LLM
                                ↓
                            Final answer
```

In [ ]:
import math
import uuid
from portkey_ai import Portkey

portkey = Portkey(api_key=PORTKEY_API_KEY)

# Define tools
TOOLS = {
    "calculator": lambda expr: str(eval(expr, {"__builtins__": {}}, {"math": math})),
    "get_weather": lambda city: f"72°F and sunny in {city} (mock data)",
}

TOOL_DEFS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a math expression. Use Python syntax.",
            "parameters": {
                "type": "object",
                "properties": {"expr": {"type": "string", "description": "Math expression"}},
                "required": ["expr"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name"}},
                "required": ["city"],
            },
        },
    },
]

print("Tools defined: calculator, get_weather")

In [ ]:
def run_agent(user_message: str, max_steps: int = 5):
    """Run a simple ReAct agent loop."""
    trace_id = f"react-{uuid.uuid4().hex[:8]}"
    messages = [
        {"role": "system", "content": "You are a helpful assistant with access to tools. Use them when needed."},
        {"role": "user", "content": user_message},
    ]

    print(f"User: {user_message}")
    print(f"Trace ID: {trace_id}")
    print("-" * 50)

    for step in range(max_steps):
        response = portkey.with_options(
            trace_id=trace_id,
            metadata={"agent": "react", "step": str(step)},
        ).chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_DEFS,
            tool_choice="auto",
            max_tokens=500,
        )

        msg = response.choices[0].message

        if not msg.tool_calls:
            print(f"\nAssistant: {msg.content}")
            return msg.content

        messages.append(msg)

        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            arg_val = list(fn_args.values())[0]

            print(f"  Step {step+1}: {fn_name}({arg_val})")

            tool_fn = TOOLS.get(fn_name)
            if tool_fn:
                result = tool_fn(arg_val)
            else:
                result = f"Unknown tool: {fn_name}"

            print(f"           → {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    print("\n(Max steps reached)")


run_agent("What is the square root of 144, and what's the weather in Tokyo?")

In [ ]:
# Another example: multi-step reasoning
run_agent("What is (25 * 4) + (100 / 5)?")

---
## 9. Reliability for Agents

Long-running agent pipelines are fragile. A single failed LLM call can crash an entire multi-step workflow. Portkey adds resilience at the gateway level:

| Feature | What it does for agents |
|---------|------------------------|
| **Fallbacks** | If GPT-4 is down mid-pipeline, auto-switch to Claude |
| **Retries** | Transient 429/500 errors are retried automatically |
| **Load balancing** | Spread agent calls across multiple API keys |
| **Timeouts** | Kill stuck LLM calls instead of hanging forever |

The agent framework doesn't need to know about any of this — Portkey handles it transparently.

In [ ]:
# 9a. Agent with fallback config
from portkey_ai import Portkey

resilient_config = {
    "retry": {"attempts": 3, "on_status_codes": [429, 500, 503]},
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": MODEL}},
        {"override_params": {"model": BACKUP_MODEL}},
    ],
}

resilient_client = Portkey(api_key=PORTKEY_API_KEY, config=resilient_config)

response = resilient_client.chat.completions.create(
    messages=[{"role": "user", "content": "Say 'resilience works!'"}],
    max_tokens=20,
)

print(f"Model used: {response.model}")
print(f"Response:   {response.choices[0].message.content}")
print("\nActive: retry (3x on 429/500/503) + fallback (primary → backup)")

In [ ]:
# 9b. Using resilient config with OpenAI Agents SDK
from agents import Agent, Runner, set_default_openai_client, set_default_openai_api
from openai import AsyncOpenAI
from portkey_ai import PORTKEY_GATEWAY_URL, createHeaders

resilient_portkey = AsyncOpenAI(
    base_url=PORTKEY_GATEWAY_URL,
    api_key=PORTKEY_API_KEY,
    default_headers=createHeaders(
        config=resilient_config,
        trace_id="resilient-agent-demo",
    ),
)

set_default_openai_client(resilient_portkey, use_for_tracing=False)
set_default_openai_api("chat_completions")

resilient_agent = Agent(
    name="Resilient Agent",
    instructions="You are helpful. Answer in one sentence.",
)

result = Runner.run_sync(resilient_agent, "What makes AI agents reliable?")
print(f"Agent: {result.final_output}")
print("\n(This agent has automatic retry + fallback via gateway config)")

In [ ]:
# 9c. Using resilient config with LangChain
from langchain_openai import ChatOpenAI
from portkey_ai import createHeaders, PORTKEY_GATEWAY_URL

resilient_llm = ChatOpenAI(
    api_key="x",
    base_url=PORTKEY_GATEWAY_URL,
    default_headers=createHeaders(
        api_key=PORTKEY_API_KEY,
        config=resilient_config,
        trace_id="langchain-resilient",
    ),
)

response = resilient_llm.invoke("Explain fallbacks in one sentence.")
print(f"Response: {response.content}")
print("\n(Same resilient config, different framework — zero code change)")

---
## 10. Observability

Every LLM call from every agent framework is logged in the Portkey dashboard with:

| Metric | Example |
|--------|---------|
| **Cost** | $0.0012 per call, $0.47 per agent run |
| **Tokens** | 150 prompt + 80 completion |
| **Latency** | 1.2s TTFB, 3.4s total |
| **Trace** | Correlated multi-step agent runs |
| **Metadata** | Framework, user ID, environment, session |
| **Model** | Which model actually served the request |

### How to use traces

Use `trace_id` to correlate all LLM calls in a single agent run:

```python
# All calls with this trace_id appear together in the dashboard
portkey.with_options(trace_id="agent-run-abc123")
```

With `metadata`, you can filter by custom dimensions:

```python
portkey.with_options(metadata={
    "framework": "crewai",
    "_user": "user-42",
    "environment": "production",
})
```

In [ ]:
import uuid
from portkey_ai import Portkey

portkey = Portkey(api_key=PORTKEY_API_KEY)

# Simulate a multi-step agent with correlated trace
trace_id = f"agent-run-{uuid.uuid4().hex[:8]}"

print(f"Running multi-step agent (trace_id: {trace_id})")
print("=" * 60)

steps = [
    {"step": "plan",      "prompt": "Plan how to explain AI Gateways in 3 bullet points."},
    {"step": "research",  "prompt": "What are the top 3 benefits of an AI Gateway?"},
    {"step": "write",     "prompt": "Write a 2-sentence summary of AI Gateways."},
]

for i, step in enumerate(steps):
    t0 = time.time()
    response = portkey.with_options(
        trace_id=trace_id,
        metadata={
            "agent_step": step["step"],
            "step_number": str(i + 1),
            "framework": "custom-react",
            "_user": "demo-user",
        },
    ).chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": step["prompt"]}],
        max_tokens=150,
    )
    elapsed = time.time() - t0

    content = response.choices[0].message.content
    tokens = response.usage.total_tokens if response.usage else "?"
    print(f"\n  Step {i+1} ({step['step']}): [{elapsed:.2f}s, {tokens} tokens]")
    print(f"  {content[:120]}{'...' if len(content) > 120 else ''}")

print("\n" + "=" * 60)
print(f"All 3 steps share trace_id: {trace_id}")
print(f"View the full trace: https://app.portkey.ai → Logs → search: {trace_id}")

---
## Recap

### Framework Integration Cheatsheet

| Framework | Integration method | Key change |
|-----------|-------------------|------------|
| **OpenAI Agents SDK** | `set_default_openai_client()` | `AsyncOpenAI(base_url=PORTKEY_GATEWAY_URL)` |
| **LangChain** | `ChatOpenAI(default_headers=...)` | `base_url=PORTKEY_GATEWAY_URL` |
| **LangGraph** | Same as LangChain | Same `ChatOpenAI` instance |
| **CrewAI** | `LLM(extra_headers=...)` | `base_url=PORTKEY_GATEWAY_URL` |
| **Strands** | `PortkeyStrands(...)` | Native adapter, no OpenAI config |
| **Google ADK** | `PortkeyAdk(...)` | Native adapter, no OpenAI config |
| **Any framework** | Auto-instrumentation | `Portkey(instrumentation=True)` |
| **Custom agents** | Raw Portkey SDK | `Portkey(api_key=...)` directly |

### What you get (for every framework)

| Capability | Without Portkey | With Portkey |
|---|---|---|
| **Observability** | Console logs, manual tracking | Full traces, cost, latency, tokens |
| **Reliability** | Agent crashes on 429/500 | Auto-retry + fallback, zero downtime |
| **Cost tracking** | No visibility | Per-call and per-agent-run cost |
| **Model flexibility** | Hard-coded provider | Swap models via config, no redeploy |
| **Guardrails** | Custom validation code | Gateway-level input/output checks |
| **Caching** | Build your own | Built-in, 20x faster repeated calls |

### Next steps

1. **Dashboard:** [app.portkey.ai](https://app.portkey.ai) — view logs, traces, and analytics
2. **Agent docs:** [docs.portkey.ai/integrations/agents](https://docs.portkey.ai/docs/integrations/agents)
3. **Auto-instrumentation:** [docs.portkey.ai/auto-instrumentation](https://docs.portkey.ai/docs/product/observability/auto-instrumentation)
4. **AI Gateway features:** See the companion notebook `portkey_ai_gateway_demo.ipynb`
5. **MCP Gateway:** See `portkey_mcp_gateway_demo.ipynb` for tool-calling through MCP